In [ ]:
from hpffbench.handler import Handler
import yaml
import logging


# paths
path_to_config = "components/handler/"

paths = {
    "path_to_benchmarks": "benchmarks",
    "path_to_tmp": "components/tmp",
    "path_to_config": "components/handler",
    "path_to_visuals": "components/visualize",
    "path_to_root": "/work/ku0598/k203191/hpffbench",
    "path_to_results": "components/results",
}


def main():

    tmp = {
        # "run01": {"X": [[1 * 134217728], [], "f8"], "Y": [[1 * 134217728], [], "f4"]},
        # "run02": {"X": [[1 * 134217728], [], "f8"]},
        "run03": {
            "X": [[1 * 134217728], []],
        },
    }

    slurm_options = """
#SBATCH --partition=compute
#SBATCH --account=
#SBATCH --constraint="[cell02]"
#SBATCH --mem=0
#SBATCH --cpu-freq=High
#SBATCH --distribution=block:cyclic
#SBATCH --time=02:00:00
#SBATCH --exclusive
#SBATCH --output=log/log-%j/log.%j.txt
#SBATCH --error=log/log-%j/log.%j.err
"""

    spack_envs = {
        "test-env-1": {
            "target": ["hdf5", {"hdf5": "subfiling"}, "netcdf4", "zarr"],
            "language": ["py", "c"],
            "packages": {
                "python": {"versions": ["3.11.14"], "fresh": True},
                "openmpi": {"versions": ["5.0.8"], "fresh": True},
                "hdf5": {
                    "versions": ["1.14.6"],
                    "variants": "~cxx~fortran+hl~ipo~java~map+mpi+shared+subfiling~szip+threadsafe+tools",
                },
                "argobots": {"versions": ["main"], "fresh": True},
                "netcdf-c": {
                    "versions": ["4.9.2"],
                    "variants": "build_system=cmake",
                },
                "py-mpi4py": {
                    "versions": ["4.0.1"],
                },
                "py-h5py": {
                    "versions": ["3.14.0"],
                },
                "py-netcdf4": {
                    "versions": ["1.7.1"],
                },
            },
            "compiler": "gcc@13",
            "additional": "pip install zarr==3.0.5 py-spy",
            # "install": True
        },
        "test-env-async": {
            "target": [{"hdf5": "async"}],
            "language": ["c"],
            "packages": {
                "python": {"versions": ["3.11.14"], "fresh": True},
                "openmpi": {
                    "versions": ["5.0.8"],
                },
                "hdf5": {
                    "versions": ["1.14.6"],
                    "variants": "~cxx~fortran+hl~ipo~java~map+mpi+shared+subfiling~szip+threadsafe+tools",
                },
                "hdf5-vol-async": {
                    "versions": ["develop"],
                },
                "argobots": {"versions": ["main"], "fresh": True},
                "netcdf-c": {
                    "versions": ["4.9.2"],
                    "variants": "build_system=cmake",
                },
                "py-mpi4py": {
                    "versions": ["4.0.1"],
                },
                "py-h5py": {
                    "versions": ["3.14.0"],
                },
                "py-netcdf4": {
                    "versions": ["1.7.1"],
                },
            },
            "compiler": "gcc@13",
            # "install": True
        },
    }

    new_setup = {
        # "formats"               : ["hdf5", "netcdf4", "zarr"],
        "formats": ["hdf5"],
        "languages": ["py"],
        "paths": paths,
        "iterations": 5,
        "runs": tmp,
        "parallel": "Both",
        "par_backend": "MPI",
        "ranks": [2],
        "nodes": [1],
        "variable_to_benchmark": ["X"],
        "only data": False,
        "use spack env": True,
        "max processes": 20,
        "slurm options": slurm_options,
        "spack env": spack_envs,
        "delete envs": False,
    }

    with open(f"{path_to_config}/config.yaml", "w") as file:
        yaml.dump(new_setup, file, sort_keys=False)

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    )
    logging.getLogger(__name__)

    Handler(path_to_config=path_to_config)


if __name__ == "__main__":
    main()